# LaPA · FP8 baseline
A100 40 GB, Python 3.13. Те же 28 вопросов, chat 5-shot, thinking off.
Выполните все ячейки сверху вниз в новой среде. ZIP скачивается в конце.
NF4-результаты встроены; загружать старый ZIP не нужно.
Загрузка FP8 проверена на A100; оценка после исправления ещё не выполнена.


In [ ]:
import hashlib
import json
import os
import signal
import subprocess
import sys
import time
import venv
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

SESSION_BUDGET_HOURS = 2  # Предел сессии, не прогноз времени.
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Выберите Python 3.13 в Colab")
if "SESSION_STARTED" not in globals():
    SESSION_STARTED = time.monotonic()
    WORK_ROOT = Path("/content/lapa_fp8")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    RUN_ROOT = WORK_ROOT / (stamp + "_" + uuid4().hex[:8])
    RESULTS_DIR = RUN_ROOT / "results"
    RESULTS_DIR.mkdir(parents=True, exist_ok=False)
print(RUN_ROOT)


## Код и фиксированные данные


In [ ]:
SOURCES = {
    'lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml': r'''group: mmlu_prox_ru_chat
task:
  - mmlu_prox_ru_chat_biology
  - mmlu_prox_ru_chat_business
  - mmlu_prox_ru_chat_chemistry
  - mmlu_prox_ru_chat_computer_science
  - mmlu_prox_ru_chat_economics
  - mmlu_prox_ru_chat_engineering
  - mmlu_prox_ru_chat_health
  - mmlu_prox_ru_chat_history
  - mmlu_prox_ru_chat_law
  - mmlu_prox_ru_chat_math
  - mmlu_prox_ru_chat_other
  - mmlu_prox_ru_chat_philosophy
  - mmlu_prox_ru_chat_physics
  - mmlu_prox_ru_chat_psychology
aggregate_metric_list:
  - aggregation: mean
    metric: exact_match
    weight_by_size: true
    filter_list: custom-extract
metadata:
  version: 0.0
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml': r'''dataset_path: li-lab/MMLU-ProX
dataset_name: ru
dataset_kwargs:
  revision: 8e6106a6c6ce1c5027e66cc338143cf997b2aa09

test_split: test
fewshot_split: validation

description: |
  Перед вами вопрос с множественным выбором.
  Выберите правильный вариант ответа.
  Финальный ответ должен заканчиваться в формате "Ответ - (X)",
  где X — латинская буква правильного варианта.

fewshot_config:
  sampler: first_n
  doc_to_text: !function utils.doc_to_text
  doc_to_target: !function utils.fewshot_target

output_type: generate_until

doc_to_text: !function utils.doc_to_text
doc_to_target: answer

filter_list:
  - name: custom-extract
    filter:
      - function: regex
        regex_pattern: 'Ответ - \(?([ABCDEFGHIJ])\)?'
      - function: take_first

generation_kwargs:
  until:
    - "</s>"
    - "Q:"
    - "Вопрос:"
    - "<|im_end|>"
  do_sample: false
  temperature: 0.0
  max_gen_toks: 2048

num_fewshot: 5

metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_case: true
    ignore_punctuation: true

metadata:
  version: 0.0
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_biology
task_alias: biology
process_docs: !function utils.process_biology
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_business
task_alias: business
process_docs: !function utils.process_business
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_chemistry
task_alias: chemistry
process_docs: !function utils.process_chemistry
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_computer_science
task_alias: computer_science
process_docs: !function utils.process_computer_science
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_economics
task_alias: economics
process_docs: !function utils.process_economics
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_engineering
task_alias: engineering
process_docs: !function utils.process_engineering
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_health
task_alias: health
process_docs: !function utils.process_health
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_history
task_alias: history
process_docs: !function utils.process_history
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_law
task_alias: law
process_docs: !function utils.process_law
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_math
task_alias: math
process_docs: !function utils.process_math
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_other
task_alias: other
process_docs: !function utils.process_other
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_philosophy
task_alias: philosophy
process_docs: !function utils.process_philosophy
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_physics
task_alias: physics
process_docs: !function utils.process_physics
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_psychology
task_alias: psychology
process_docs: !function utils.process_psychology
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/utils.py': r'''from functools import partial

CHOICES = list("ABCDEFGHIJ")
MAX_OPTIONS = 10


def doc_to_text(example):
    prompt = "Вопрос:\n"
    prompt += example["question"] + "\n"
    prompt += "Варианты:\n"

    for i in range(MAX_OPTIONS):
        option = example[f"option_{i}"]
        if option is not None:
            prompt += f"{CHOICES[i]}. {option}\n"

    return prompt


def fewshot_target(example):
    return f"Ответ - ({example['answer']})"


def process_docs(dataset, subject):
    return dataset.filter(lambda x: x["category"] == subject)


process_biology = partial(process_docs, subject="biology")
process_business = partial(process_docs, subject="business")
process_chemistry = partial(process_docs, subject="chemistry")
process_computer_science = partial(
    process_docs, subject="computer science"
)
process_economics = partial(process_docs, subject="economics")
process_engineering = partial(process_docs, subject="engineering")
process_health = partial(process_docs, subject="health")
process_history = partial(process_docs, subject="history")
process_law = partial(process_docs, subject="law")
process_math = partial(process_docs, subject="math")
process_other = partial(process_docs, subject="other")
process_philosophy = partial(process_docs, subject="philosophy")
process_physics = partial(process_docs, subject="physics")
process_psychology = partial(process_docs, subject="psychology")
''',
    'nf4_reference.json': r'''{
  "archive_sha256": "7d857e967f4d263114cb4b3247647b3036834582f5ea4d9cc027f52d185e826d",
  "selection": {
    "protocol": "chat-5shot-baseline-v1",
    "dataset_revision": "8e6106a6c6ce1c5027e66cc338143cf997b2aa09",
    "model_revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
    "harness_revision": "d6de81643928d653435c431bae19945d41d32520",
    "sources": {
      "requirements-eval.txt": "66c2e8fd224964045b54e1617a290871d99a2295805386942c9645a5d0e6ae8b",
      "eval_support/__init__.py": "78d7a894f2ba87185a30a183c3696b907b0b0e04f8ccfda7baca3c23c258d849",
      "eval_support/standalone.py": "98208d64d375a03f484836cfb9e92d28e422a66f109d270862ba22637a5e43e3",
      "eval_support/mmluprox.py": "2a044bd537cc1277c2f997b06eb8bf5f52861655db0307501e7d7e036e3d634e",
      "lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml": "852bf00302ac68b796a07121646afeb1c054f165b1b7e1585c89a7a549193669",
      "lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml": "ba71b82a8872d81e841ca566cf1228fbf6bac288963c40669112c4d241156093",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml": "5c5652af779b6f76263c8b2c50d1e556b49ed9ae135a569de7a729f819d2db23",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml": "c3ee2987b8742f81fa4eb59262d3bd247a65efa6a9cb2a75ed7d75427bf08d75",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml": "5be779e17271bb81d0ebe8702a24c3ac399287d0df4836ab81524ea6a098257a",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml": "1816af75a73635c7705c35aef3072f15a22ee54aa305819880927dcf7e3a9ae5",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml": "ef1fcf83eadc67e89bb3b4381bad0eac16585c3bfc266bf5d124f99d4030d487",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml": "fe1bcfa0009c42c7998c37ede092bcf3bb9b896dbbc5885980f92a89ea8f05b5",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml": "80e6d0e0613752a2454df265c58f34f9448c4791e347893abda72305e6fbc64a",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml": "047c600f10c0b38e0b3eff2c428079793c3d89ecd2bcc1b43019df6c83f84956",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml": "7679544fafe2ada9c1e667e4adc13db1f319f420ecb97908ef48198737ba2289",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml": "af96df4b28b32daa1846bc0f29264709ebe2e7a7c4789f137fa900ae61d49694",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml": "87a294c66ccf38044a6016ad0ae0c5cb5a4c3f092ebd2c40dbdc6c4de3ff1580",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml": "fb0599414d22276942170f10fc4238a590e4981d04be08ceae9202af54fe0fe2",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml": "bfb64e6779bef91b5092dea22efee61f8ebf1e7888861558bd308a3a6d4e7196",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml": "cb4802cc2f6f07283d8a433cd55512d436a34d653d6ee5d09f1e0cf561e6a751",
      "lm_eval_tasks/mmlu_prox_ru_chat/utils.py": "99d10756ca88eee8fa3ff9fce3a4d4efb746b17a7a9b9feb7fcfbb76b15e5f3a",
      "protocol.md": "872a181f30d8469c7bb9ded627ab3e7a0a51b2d153c5ea484a1c080a827acfd0"
    },
    "seed": 20260926,
    "per_subject": 2,
    "items": [
      {
        "subject": "biology",
        "question_id": 3231,
        "index": 615,
        "sha256": "f702a8f13cc82aea4ee3be52d7859d0c1f805054423147582acc082720e9dfcd"
      },
      {
        "subject": "biology",
        "question_id": 3040,
        "index": 424,
        "sha256": "340598dc5043f32537a585bafd5083c544fa9a521d303f6fd63859cd5fe2d3ba"
      },
      {
        "subject": "business",
        "question_id": 746,
        "index": 676,
        "sha256": "d89d7bbe5af6d88b68a33d266d060370f5f52794a4901635823a8f9d4da697fa"
      },
      {
        "subject": "business",
        "question_id": 209,
        "index": 139,
        "sha256": "2587ef5ee8c725c633290431c50446dda013911ece5c8a9dc3d4e25c9956657d"
      },
      {
        "subject": "chemistry",
        "question_id": 3794,
        "index": 461,
        "sha256": "81de309a6f3c2a04cae6b668cc8c1308277186ae2edc0666bc4807543a9de2cb"
      },
      {
        "subject": "chemistry",
        "question_id": 4239,
        "index": 906,
        "sha256": "82493408817cf9cae3df62c968a304773863a752dcb6c67022ab275ad1874fe5"
      },
      {
        "subject": "computer_science",
        "question_id": 10221,
        "index": 270,
        "sha256": "59d6c3eb6cf32c8d104762e0cc67a6dabf71ba2ea7900fe73e068088cdd51fba"
      },
      {
        "subject": "computer_science",
        "question_id": 9981,
        "index": 30,
        "sha256": "db291da535c550499f1d539c4d4bafc2c385fad36dab667cc675a1bbe90b7b7a"
      },
      {
        "subject": "economics",
        "question_id": 6540,
        "index": 83,
        "sha256": "0addc0fb371b81b958ef6a13199631741bbfda8c62dc2676c6d8ae65f5df4dd6"
      },
      {
        "subject": "economics",
        "question_id": 6815,
        "index": 358,
        "sha256": "c6f18a65d8c361ef57dab75c8f2ea23966bff2efb425686a359360e599de1e56"
      },
      {
        "subject": "engineering",
        "question_id": 11735,
        "index": 875,
        "sha256": "b829c72356ad369b2d8336c2e1e8e7cb968e10396f01f0c660a1ddda696d2e84"
      },
      {
        "subject": "engineering",
        "question_id": 11767,
        "index": 907,
        "sha256": "c8930281464d36c76a1d263c78f4cb90efbcb13895a5833e6646d78dbe761e2a"
      },
      {
        "subject": "health",
        "question_id": 6285,
        "index": 515,
        "sha256": "4cc725c951ac67fedfa52630cb56012e60ff08735a2e6660a727db13893d1e17"
      },
      {
        "subject": "health",
        "question_id": 5845,
        "index": 75,
        "sha256": "1bb13ed50b0943fd0d9618cfff1d7ca708766a149a722554d80c5689032a1178"
      },
      {
        "subject": "history",
        "question_id": 4592,
        "index": 127,
        "sha256": "aea29f221253bb42e4005a2a976e88e4a75c883585833408e573e4d01a169ee3"
      },
      {
        "subject": "history",
        "question_id": 4666,
        "index": 201,
        "sha256": "a864ef88ee72ff7216a8f7ff0388d7d113ae33de959194805ab98643cb7513f5"
      },
      {
        "subject": "law",
        "question_id": 1014,
        "index": 155,
        "sha256": "85d003d7f48c8d7609fd7048f7cc134ef8f6342ad51fb65b7ad9651902a748a1"
      },
      {
        "subject": "law",
        "question_id": 1246,
        "index": 387,
        "sha256": "f067cf34ca7ed7807fe3feb0cae6335db9022cb5f9ba3362d985f9c824de364a"
      },
      {
        "subject": "math",
        "question_id": 8530,
        "index": 1229,
        "sha256": "dfe9945513e7e4c568a63671f03aea535542c406d331afa619be7aef1f5de21f"
      },
      {
        "subject": "math",
        "question_id": 7777,
        "index": 476,
        "sha256": "92a5cf5e03adae55d07009bcfc66c4e4b1dc76d188b3810f46a5895968df2496"
      },
      {
        "subject": "other",
        "question_id": 5546,
        "index": 700,
        "sha256": "ad4fadef3d666d7ae80a703c6a412ef51809e95edc35cabeadadbe95bb7d6af4"
      },
      {
        "subject": "other",
        "question_id": 4871,
        "index": 25,
        "sha256": "3818fecf26f6e8cc043dc63e33a5f9d29493115cab6765eb9edd21abb9f8e249"
      },
      {
        "subject": "philosophy",
        "question_id": 10438,
        "index": 77,
        "sha256": "0f172bf7e974ed46d1e356a751205e8abafc68bdf4cb85b9ce6c5546321b0691"
      },
      {
        "subject": "philosophy",
        "question_id": 10591,
        "index": 230,
        "sha256": "3c59f4b6e06df5e6e3e6e43122f421c1ec259681c05c55c051f72775cda92bd7"
      },
      {
        "subject": "physics",
        "question_id": 8823,
        "index": 171,
        "sha256": "1abd9ab6726e2086bb1c5a168f16fae78082b1e014be59717751d220d6507912"
      },
      {
        "subject": "physics",
        "question_id": 9484,
        "index": 832,
        "sha256": "7688b36ec3d1b747f81007bb6d9823e12b81b9f7a3489507df0a05259d97e717"
      },
      {
        "subject": "psychology",
        "question_id": 1925,
        "index": 107,
        "sha256": "0974c1b241dcf818aa40e2866075bd22ae9c7c814920c939e16dd31607373927"
      },
      {
        "subject": "psychology",
        "question_id": 2215,
        "index": 397,
        "sha256": "df2b969bc9e353641a711b1220575a8057af3144d4c84ce5eacef061022d275f"
      }
    ],
    "excluded_development": [
      {
        "subject": "biology",
        "question_id": 3273,
        "index": 657,
        "sha256": "e9f216bfdd60cbb2cb91a2654cff5c21497bb062046a7b6eae20573613f376d3"
      },
      {
        "subject": "business",
        "question_id": 478,
        "index": 408,
        "sha256": "0ceee92224cb1212dc5cc5d8f91687571d7210d22afceed0663b56d8db2e7c5a"
      },
      {
        "subject": "chemistry",
        "question_id": 3881,
        "index": 548,
        "sha256": "eb8562cf90fef498a87b06795c8b43384e1ac07ee09f5543abe081bcc9bb2f89"
      },
      {
        "subject": "computer_science",
        "question_id": 10044,
        "index": 93,
        "sha256": "1f5df94a308a053e1f81f655eda8c0434adfd8ff40fba7566ce32b93eae0b6f8"
      },
      {
        "subject": "economics",
        "question_id": 6749,
        "index": 292,
        "sha256": "78f865f5d671bc77f09a75aebf042b929ff023f842419c1abfb938fc46349b44"
      },
      {
        "subject": "engineering",
        "question_id": 11659,
        "index": 799,
        "sha256": "fdd3f999058ebfeb59091471ac5c0f722849b01853b73ddbe5220cc183a78e66"
      },
      {
        "subject": "health",
        "question_id": 6021,
        "index": 251,
        "sha256": "e9cd343801e6cf3da9c676c88f67ae0b9ab673f59152975528d68599a7f50151"
      },
      {
        "subject": "history",
        "question_id": 4806,
        "index": 341,
        "sha256": "bd97117397a29c46bdd8685d59509102e9f0785ea21c1cc61c64b2cd69ee27d1"
      },
      {
        "subject": "law",
        "question_id": 1080,
        "index": 221,
        "sha256": "fce9733331ef41251f444ea2c51c46e3c783eb494636cc63819fca8b82e132dd"
      },
      {
        "subject": "math",
        "question_id": 7973,
        "index": 672,
        "sha256": "dbe21c95dfe7ca0bd3afcbd3a13a3dd1cf0c2f98656e4f551cb59eb87c983f8c"
      },
      {
        "subject": "other",
        "question_id": 5375,
        "index": 529,
        "sha256": "3ca03e5c35c22c3d3cd2f2b40c016848f712ee9af39c4756f47c25a2ab74dd32"
      },
      {
        "subject": "philosophy",
        "question_id": 10485,
        "index": 124,
        "sha256": "fbac2683998236a4a08798844f53c06d42740f906d53b3903c095bf04cb72f4f"
      },
      {
        "subject": "physics",
        "question_id": 9502,
        "index": 850,
        "sha256": "92b10e8627876818b3beaeb68b95e3ee99707439bd7249166541eee356e3c318"
      },
      {
        "subject": "psychology",
        "question_id": 2419,
        "index": 601,
        "sha256": "0f32df6e46138d7de4fd61fbfa975789b8dce653e5fc84a2aa874d1c12de8760"
      }
    ],
    "shots": 5,
    "chat": true,
    "thinking": false,
    "max_gen_toks": 2048,
    "batch_size": 1,
    "session_budget_seconds": 21600,
    "question_budget_seconds": 600,
    "prior_gpu_seconds": 4289,
    "note": "Preliminary stratified baseline, not a full benchmark"
  },
  "records": [
    {
      "subject": "biology",
      "question_id": 3231,
      "index": 615,
      "sha256": "f702a8f13cc82aea4ee3be52d7859d0c1f805054423147582acc082720e9dfcd",
      "nf4_correct": 0.0,
      "nf4_answer": "H",
      "input_ids_sha256": "38deb73bdca3c4d9cef9f3e3185967d391a792d752bcd05202df6bb7ee57b00c"
    },
    {
      "subject": "biology",
      "question_id": 3040,
      "index": 424,
      "sha256": "340598dc5043f32537a585bafd5083c544fa9a521d303f6fd63859cd5fe2d3ba",
      "nf4_correct": 1.0,
      "nf4_answer": "F",
      "input_ids_sha256": "8d5168e41767c695ccd289d89415f2fabac2a269246addb6e7dced82dc58c8b8"
    },
    {
      "subject": "business",
      "question_id": 746,
      "index": 676,
      "sha256": "d89d7bbe5af6d88b68a33d266d060370f5f52794a4901635823a8f9d4da697fa",
      "nf4_correct": 1.0,
      "nf4_answer": "H",
      "input_ids_sha256": "5ecbd2204833423c2f62b88792d8e5cb3727ff9964a4c87754e459eb9a1daccc"
    },
    {
      "subject": "business",
      "question_id": 209,
      "index": 139,
      "sha256": "2587ef5ee8c725c633290431c50446dda013911ece5c8a9dc3d4e25c9956657d",
      "nf4_correct": 1.0,
      "nf4_answer": "J",
      "input_ids_sha256": "213282bcbf1c67e0259460b33a53fc77a1f968ffda6f111f2371fa38e800e65d"
    },
    {
      "subject": "chemistry",
      "question_id": 3794,
      "index": 461,
      "sha256": "81de309a6f3c2a04cae6b668cc8c1308277186ae2edc0666bc4807543a9de2cb",
      "nf4_correct": 0.0,
      "nf4_answer": "F",
      "input_ids_sha256": "e90fee2a1a7f5f793738c9ec85b587dd799fd23b162de8876613dd648e5d385e"
    },
    {
      "subject": "chemistry",
      "question_id": 4239,
      "index": 906,
      "sha256": "82493408817cf9cae3df62c968a304773863a752dcb6c67022ab275ad1874fe5",
      "nf4_correct": 1.0,
      "nf4_answer": "G",
      "input_ids_sha256": "5fc2a02de58b9b61d2a65438d4f93e1955501cc93859dcc0a0bb1a80d95ef53f"
    },
    {
      "subject": "computer_science",
      "question_id": 10221,
      "index": 270,
      "sha256": "59d6c3eb6cf32c8d104762e0cc67a6dabf71ba2ea7900fe73e068088cdd51fba",
      "nf4_correct": 1.0,
      "nf4_answer": "F",
      "input_ids_sha256": "3d61a1689ee165b2a7ec0149311b8ead01d59691f56f0756edc05f0c22b65304"
    },
    {
      "subject": "computer_science",
      "question_id": 9981,
      "index": 30,
      "sha256": "db291da535c550499f1d539c4d4bafc2c385fad36dab667cc675a1bbe90b7b7a",
      "nf4_correct": 1.0,
      "nf4_answer": "G",
      "input_ids_sha256": "b71c6c7ceab6bb5923053831b37a21db906e801effa19bee1e0473b12ef3ca01"
    },
    {
      "subject": "economics",
      "question_id": 6540,
      "index": 83,
      "sha256": "0addc0fb371b81b958ef6a13199631741bbfda8c62dc2676c6d8ae65f5df4dd6",
      "nf4_correct": 1.0,
      "nf4_answer": "J",
      "input_ids_sha256": "1ff7346f08940d3c7f0a8fb3d2d36d993f989d548ff817c92e5b3abe5379bee3"
    },
    {
      "subject": "economics",
      "question_id": 6815,
      "index": 358,
      "sha256": "c6f18a65d8c361ef57dab75c8f2ea23966bff2efb425686a359360e599de1e56",
      "nf4_correct": 1.0,
      "nf4_answer": "D",
      "input_ids_sha256": "45c628a4ad90cbd2817c769f1cd9202e7172d6dbf09bb28b0a3043efceb1dc6d"
    },
    {
      "subject": "engineering",
      "question_id": 11735,
      "index": 875,
      "sha256": "b829c72356ad369b2d8336c2e1e8e7cb968e10396f01f0c660a1ddda696d2e84",
      "nf4_correct": 1.0,
      "nf4_answer": "D",
      "input_ids_sha256": "be63e4ac50bccac7ad3f135c884616154c2e4e00e6dc4664c1e26662028f6d88"
    },
    {
      "subject": "engineering",
      "question_id": 11767,
      "index": 907,
      "sha256": "c8930281464d36c76a1d263c78f4cb90efbcb13895a5833e6646d78dbe761e2a",
      "nf4_correct": 0.0,
      "nf4_answer": "[invalid]",
      "input_ids_sha256": "40989653eeef22352aa2126bef9522f98cb7d71c40a70a77cd9167243617f450"
    },
    {
      "subject": "health",
      "question_id": 6285,
      "index": 515,
      "sha256": "4cc725c951ac67fedfa52630cb56012e60ff08735a2e6660a727db13893d1e17",
      "nf4_correct": 1.0,
      "nf4_answer": "J",
      "input_ids_sha256": "e0ebd02b4b313f40836c0e32fd3c62a28b59b128354862396448987c0baff71c"
    },
    {
      "subject": "health",
      "question_id": 5845,
      "index": 75,
      "sha256": "1bb13ed50b0943fd0d9618cfff1d7ca708766a149a722554d80c5689032a1178",
      "nf4_correct": 1.0,
      "nf4_answer": "H",
      "input_ids_sha256": "d331733e7b91da11ef06391559394d9c611b3c3d8a7f3d35bed2fbb5a7300e46"
    },
    {
      "subject": "history",
      "question_id": 4592,
      "index": 127,
      "sha256": "aea29f221253bb42e4005a2a976e88e4a75c883585833408e573e4d01a169ee3",
      "nf4_correct": 0.0,
      "nf4_answer": "J",
      "input_ids_sha256": "4e2541899ba43b3b1f7ff1aec28f0fad793a2ef8a1c3e9bf95d87cbd4e3c5a75"
    },
    {
      "subject": "history",
      "question_id": 4666,
      "index": 201,
      "sha256": "a864ef88ee72ff7216a8f7ff0388d7d113ae33de959194805ab98643cb7513f5",
      "nf4_correct": 1.0,
      "nf4_answer": "C",
      "input_ids_sha256": "0ddd4057ea0f63824f62b5665ea1a4213075ea68627cabb781bd32aee6c3cb7e"
    },
    {
      "subject": "law",
      "question_id": 1014,
      "index": 155,
      "sha256": "85d003d7f48c8d7609fd7048f7cc134ef8f6342ad51fb65b7ad9651902a748a1",
      "nf4_correct": 1.0,
      "nf4_answer": "B",
      "input_ids_sha256": "e9fa7c50b298ff32d00f79798392a169ab67ca2e8e2df505da81db8fa241092c"
    },
    {
      "subject": "law",
      "question_id": 1246,
      "index": 387,
      "sha256": "f067cf34ca7ed7807fe3feb0cae6335db9022cb5f9ba3362d985f9c824de364a",
      "nf4_correct": 0.0,
      "nf4_answer": "E",
      "input_ids_sha256": "bc5155deee3986f3130ac6a5b3d8bbaf6661f7b583c6530d242ef9e331e49ae5"
    },
    {
      "subject": "math",
      "question_id": 8530,
      "index": 1229,
      "sha256": "dfe9945513e7e4c568a63671f03aea535542c406d331afa619be7aef1f5de21f",
      "nf4_correct": 1.0,
      "nf4_answer": "G",
      "input_ids_sha256": "7129c47a9b5a8f8bb7f8a9e96e6b51a3b942252bd9b7c50ec36ff6a69518ffaf"
    },
    {
      "subject": "math",
      "question_id": 7777,
      "index": 476,
      "sha256": "92a5cf5e03adae55d07009bcfc66c4e4b1dc76d188b3810f46a5895968df2496",
      "nf4_correct": 1.0,
      "nf4_answer": "I",
      "input_ids_sha256": "88b03eb0bdb59e9a11efe17a48de53b8ca3ce88706a8738fba6a1361217f553c"
    },
    {
      "subject": "other",
      "question_id": 5546,
      "index": 700,
      "sha256": "ad4fadef3d666d7ae80a703c6a412ef51809e95edc35cabeadadbe95bb7d6af4",
      "nf4_correct": 0.0,
      "nf4_answer": "F",
      "input_ids_sha256": "2440466bccb555e5b13958e8cf95ad5a002f7218edb858d6e062dab5d7f1d161"
    },
    {
      "subject": "other",
      "question_id": 4871,
      "index": 25,
      "sha256": "3818fecf26f6e8cc043dc63e33a5f9d29493115cab6765eb9edd21abb9f8e249",
      "nf4_correct": 0.0,
      "nf4_answer": "C",
      "input_ids_sha256": "d645d818f4ab89fc836272fdb6560241d8f85eccd1b1d8844fe5eb7ebd5a6844"
    },
    {
      "subject": "philosophy",
      "question_id": 10438,
      "index": 77,
      "sha256": "0f172bf7e974ed46d1e356a751205e8abafc68bdf4cb85b9ce6c5546321b0691",
      "nf4_correct": 1.0,
      "nf4_answer": "B",
      "input_ids_sha256": "96c6fc9413a47280c5e3a0e96db5e53482880df4617a97990ec1aaae7737d89e"
    },
    {
      "subject": "philosophy",
      "question_id": 10591,
      "index": 230,
      "sha256": "3c59f4b6e06df5e6e3e6e43122f421c1ec259681c05c55c051f72775cda92bd7",
      "nf4_correct": 1.0,
      "nf4_answer": "G",
      "input_ids_sha256": "5e3a02f5f855a7c262ff641d0ee2ae38f92af87a14a9f09ebd2c7fbf0c7eeb42"
    },
    {
      "subject": "physics",
      "question_id": 8823,
      "index": 171,
      "sha256": "1abd9ab6726e2086bb1c5a168f16fae78082b1e014be59717751d220d6507912",
      "nf4_correct": 1.0,
      "nf4_answer": "J",
      "input_ids_sha256": "c457f9302ceb6b47ccc896f1d65218db318695f9de5ba935d8cb3581e08029b2"
    },
    {
      "subject": "physics",
      "question_id": 9484,
      "index": 832,
      "sha256": "7688b36ec3d1b747f81007bb6d9823e12b81b9f7a3489507df0a05259d97e717",
      "nf4_correct": 1.0,
      "nf4_answer": "A",
      "input_ids_sha256": "bc79619f0e2e586762b13131e98bf08eac725b7b8711ef7bbb24a5309b8ac369"
    },
    {
      "subject": "psychology",
      "question_id": 1925,
      "index": 107,
      "sha256": "0974c1b241dcf818aa40e2866075bd22ae9c7c814920c939e16dd31607373927",
      "nf4_correct": 1.0,
      "nf4_answer": "B",
      "input_ids_sha256": "d64e5d5627ea27538776de70c54cf9dc21aef5d1ed7741ed51e6932c9170728d"
    },
    {
      "subject": "psychology",
      "question_id": 2215,
      "index": 397,
      "sha256": "df2b969bc9e353641a711b1220575a8057af3144d4c84ce5eacef061022d275f",
      "nf4_correct": 1.0,
      "nf4_answer": "G",
      "input_ids_sha256": "90fa34fd1553bdf5a398acb5aab912a48750ecd37b598a3d847bd409b2d31bbf"
    }
  ]
}
''',
    'requirements.txt': r'''--index-url https://pypi.org/simple
--extra-index-url https://download.pytorch.org/whl/cu129

absl-py==2.5.0
agent-detector==2.0.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.4
aiosignal==1.4.0
annotated-doc==0.0.5
annotated-types==0.8.0
anthropic==1.11.0
anyio==4.15.1
apache-tvm-ffi==0.1.9
astor==0.8.1
attrs==26.1.0
blake3==1.0.10
cachetools==7.2.1
cbor2==6.1.5
certifi==2026.7.22
cffi==2.1.1
chardet==6.0.0.post1
charset-normalizer==3.5.2
click==8.5.0
cloudpickle==3.1.2
colorama==0.4.6
compressed-tensors==0.15.0.1
cryptography==50.0.2
cuda-bindings==12.9.9
cuda-pathfinder==1.8.3
cuda-tile==1.6.0
cuda-toolkit==12.9.1
dataproperty==1.1.1
datasets==4.6.1
defusedxml==0.7.1
depyf==0.20.0
detect-installer==0.2.1
dill==0.4.0
diskcache==5.6.3
dnspython==2.8.0
docstring-parser==0.18.0
einops==0.8.2
email-validator==2.3.0
evaluate==0.4.6
fastapi==0.142.2
fastapi-cli==0.0.32
fastapi-cloud-cli==0.26.0
fastar==0.12.0
fastsafetensors==0.4.0
filelock==4.0.12
flashinfer-cubin==0.6.8.post1
flashinfer-python==0.6.8.post1
frozenlist==1.8.0
fsspec==2026.2.0
gguf==0.19.0
googleapis-common-protos==1.75.5
grpcio==1.84.0
h11==0.16.0
hf-xet==1.7.0
httpcore==1.0.9
httpcore2==2.13.1
httptools==0.8.0
httpx==0.28.1
httpx2==2.13.1
huggingface-hub==1.29.0
idna==3.20
ijson==3.5.1
interegular==0.3.3
jinja2==3.1.6
jiter==0.17.0
jmespath==1.1.0
joblib==1.6.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
lark==1.2.2
llguidance==1.3.0
llvmlite==0.47.0
lm-eval @ git+https://github.com/EleutherAI/lm-evaluation-harness.git@d6de81643928d653435c431bae19945d41d32520
lm-format-enforcer==0.11.3
loguru==0.7.3
lxml==6.1.3
markdown-it-py==4.2.0
markupsafe==3.0.4
mbstrdecoder==1.1.5
mcp==2.3.0
mcp-types==2.3.0
mdurl==0.1.2
mistral-common==1.12.0
ml-dtypes==0.6.0
model-hosting-container-standards==0.1.16
more-itertools==11.1.0
mpmath==1.3.0
msgspec==0.22.0
multidict==6.9.1
multiprocess==0.70.18
narwhals==2.26.0
networkx==3.7
ninja==1.13.2
nltk==3.10.3
numba==0.65.0
numpy==2.4.6
nvidia-cublas-cu12==12.9.1.4
nvidia-cuda-cupti-cu12==12.9.79
nvidia-cuda-nvrtc-cu12==12.9.86
nvidia-cuda-runtime-cu12==12.9.79
nvidia-cudnn-cu12==9.17.1.4
nvidia-cudnn-frontend==1.18.0
nvidia-cufft-cu12==11.4.1.4
nvidia-cufile-cu12==1.14.1.1
nvidia-curand-cu12==10.3.10.19
nvidia-cusolver-cu12==11.7.5.82
nvidia-cusparse-cu12==12.5.10.65
nvidia-cusparselt-cu12==0.7.1
nvidia-cutlass-dsl==4.8.0
nvidia-cutlass-dsl-libs-base==4.8.0
nvidia-cutlass-dsl-libs-core==4.8.0
nvidia-cutlass-dsl-libs-cu12==4.8.0
nvidia-ml-py==13.615.71
nvidia-nccl-cu12==2.28.9
nvidia-nvjitlink-cu12==12.9.86
nvidia-nvshmem-cu12==3.4.5
nvidia-nvtx-cu12==12.9.79
openai==3.26.0
openai-harmony==0.0.8
opencv-python-headless==5.0.0.93
opentelemetry-api==1.45.1
opentelemetry-exporter-http-transport==0.66b1
opentelemetry-exporter-otlp==1.45.1
opentelemetry-exporter-otlp-common==0.66b1
opentelemetry-exporter-otlp-proto-common==1.45.1
opentelemetry-exporter-otlp-proto-grpc==1.45.1
opentelemetry-exporter-otlp-proto-http==1.45.1
opentelemetry-proto==1.45.1
opentelemetry-sdk==1.45.1
opentelemetry-semantic-conventions==0.66b1
opentelemetry-semantic-conventions-ai==0.5.1
outlines-core==0.2.14
packaging==26.3
pandas==3.0.6
partial-json-parser==0.2.1.1.post7
pathvalidate==3.3.1
pillow==12.3.0
portalocker==4.4.0
prometheus-client==0.26.0
prometheus-fastapi-instrumentator==8.1.0
propcache==0.5.4
protobuf==7.36.2
psutil==7.2.2
py-cpuinfo==9.0.0
pyarrow==25.0.1
pybase64==1.5.1
pycountry==26.2.16
pycparser==3.0
pydantic==2.13.5
pydantic-core==2.46.5
pydantic-extra-types==2.11.1
pydantic-settings==2.15.0
pygments==2.21.0
pyjwt==2.15.1
pytablewriter==1.2.1
python-dateutil==2.9.0.post0
python-dotenv==1.2.4
python-json-logger==4.2.0
python-multipart==0.0.32
pytz==2026.5
pyyaml==6.0.3
pyzmq==27.2.0
quack-kernels==0.6.5
referencing==0.37.0
regex==2026.9.29
requests==2.34.2
rich==15.0.0
rich-toolkit==0.20.5
rignore==0.8.1
rouge-score==0.1.2
rpds-py==2026.9.1
sacrebleu==2.6.0
safetensors==0.8.0
scikit-learn==1.9.1
scipy==1.18.1
sentencepiece==0.2.2
sentry-sdk==2.71.0
setproctitle==1.3.8
setuptools==80.10.2
shellingham==1.5.4
six==1.17.0
sniffio==1.3.1
sqlitedict==2.1.0
sse-starlette==3.5.0
starlette==1.7.0
supervisor==4.3.0
sympy==1.14.0
tabledata==1.3.5
tabulate==0.10.0
tcolorpy==0.1.7
threadpoolctl==3.7.0
tiktoken==0.14.0
tilelang==0.1.9
tokenizers==0.22.2
torch==2.11.0+cu129
torch-c-dlpack-ext==0.1.5
torchaudio==2.11.0+cu129
torchvision==0.26.0+cu129
tqdm==4.70.1
transformers==5.5.1
triton==3.6.0
truststore==0.10.4
typepy==1.3.5
typer==0.27.3
typing-extensions==4.16.0
typing-inspection==0.4.4
urllib3==2.8.0
uvicorn==0.54.0
uvloop==0.23.0
vllm @ https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0%2Bcu129-cp38-abi3-manylinux_2_31_x86_64.whl#sha256=b687351527b0bca7f6b0f367b6c93499e19b06f0c9a05b01084c043f7ec50ce7
watchfiles==1.3.0
websockets==17.2
word2number==1.1
xgrammar==0.2.3
xxhash==4.0.1
yarl==1.25.1
z3-solver==4.15.4.0
''',
    'baseline.py': r'''"""FP8 baseline; scoring and generation belong to lm-eval."""

import hashlib
import json
import time
import traceback
from pathlib import Path

import lm_eval
from lm_eval.models.vllm_causallms import VLLM
from lm_eval.utils import handle_non_serializable

DATA_REVISION = "8e6106a6c6ce1c5027e66cc338143cf997b2aa09"
MODEL_REVISION = "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a"
BASE_REVISION = "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0"
HARNESS_REVISION = "d6de81643928d653435c431bae19945d41d32520"
MODEL_ID = "Qwen/Qwen3.8-27B-FP8"


class AuditedVLLM(VLLM):
    """Record native outputs and reject changed or truncated inputs."""

    raw_path = None
    deadline = None
    expected_input_hash = None

    def generate_until(self, requests, **kwargs):
        for request in requests:
            tokens = self.tok_encode(request.args[0])
            limit = request.args[1].get("max_gen_toks", 2048)
            if len(tokens) + limit > self.max_length:
                raise ValueError("Input exceeds context; no truncation")
            digest = hashlib.sha256(json.dumps(tokens).encode()).hexdigest()
            if self.expected_input_hash and digest != self.expected_input_hash:
                raise ValueError("Prompt tokens differ from NF4 baseline")
        return super().generate_until(requests, **kwargs)

    def _model_generate(self, requests, generate=False, sampling_params=None):
        started = time.monotonic()
        outputs = super()._model_generate(
            requests, generate=generate, sampling_params=sampling_params,
        )
        elapsed = time.monotonic() - started
        if generate and self.raw_path is not None:
            with self.raw_path.open("a", encoding="utf-8") as stream:
                for output in outputs:
                    answer = output.outputs[0]
                    record = {
                        "input_ids": list(output.prompt_token_ids),
                        "generated_ids": list(answer.token_ids),
                        "raw_text": answer.text,
                        "finish_reason": answer.finish_reason,
                        "stop_reason": answer.stop_reason,
                        "generation_seconds": elapsed,
                        "note": "vLLM wall time; not paired NF4 speedup",
                    }
                    stream.write(json.dumps(record, ensure_ascii=False))
                    stream.write("\n")
        return outputs


def save_json(path, value):
    """Never overwrite a previous result."""
    with Path(path).open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2,
                  default=handle_non_serializable)


def seal(directory):
    """Hash saved files, including partial raw output after failure."""
    directory = Path(directory)
    files = {}
    for path in sorted(directory.rglob("*")):
        if path.is_file() and path.name != "manifest.json":
            with path.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            files[str(path.relative_to(directory))] = digest
    save_json(directory / "manifest.json", files)


def run_eval(lm, task_manager, directory, name, task, sample_index,
             document_hash):
    """Evaluate chat 5-shot without native thinking."""
    run_dir = Path(directory) / name
    run_dir.mkdir(parents=True, exist_ok=False)
    lm.enable_thinking = False
    lm.think_end_token = None
    lm.raw_path = run_dir / "raw_generations.jsonl"
    started = time.monotonic()
    save_json(run_dir / "settings.json", {
        "task": task, "shots": 5, "chat": True,
        "thinking": False, "limit": None,
        "dataset_revision": DATA_REVISION,
        "model_revision": MODEL_REVISION,
        "harness_revision": HARNESS_REVISION,
        "sample_index": sample_index, "document_hash": document_hash,
        "debug": False, "split": "test",
    })
    try:
        # Pinned harness factory propagates revision before dataset loading.
        # A bare {task: group_name} is an inline task, not a group reference.
        overrides = {"dataset_kwargs": {"revision": DATA_REVISION}}
        prepared_task = task_manager._factory.build(
            task_manager.task_index[task],
            overrides=overrides,
            registry=task_manager.task_index,
        )
        document = prepared_task.test_docs()[sample_index]
        if digest_json(document) != document_hash:
            raise ValueError("Selected document differs from the plan")
        result = lm_eval.simple_evaluate(
            model=lm,
            tasks=[prepared_task],
            task_manager=task_manager,
            num_fewshot=5,
            apply_chat_template=True,
            fewshot_as_multiturn=True,
            samples={task: [sample_index]},
            log_samples=True,
            bootstrap_iters=0,
            random_seed=0,
            numpy_random_seed=1234,
            torch_random_seed=1234,
            fewshot_random_seed=1234,
        )
        save_json(run_dir / f"{name}.json", result)
        save_json(run_dir / "timing.json", {
            "wall_seconds": time.monotonic() - started,
            "note": "Evaluation wall time, not paired inference E2E",
        })
        seal(run_dir)
        save_json(run_dir / "completed.json", {"status": "completed"})
        return result
    except BaseException:
        original_traceback = traceback.format_exc()
        try:
            save_json(run_dir / "failure.json", {
                "traceback": original_traceback,
                "wall_seconds": time.monotonic() - started,
            })
        except OSError as error:
            print("Could not save failure diagnostics:", error)
        raise
    finally:
        lm.raw_path = None


def digest_json(value):
    """Stable identity for documents and frozen plans."""
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False,
                         separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def verify_run(directory):
    """Accept only a sealed, completed harness result."""
    directory = Path(directory)
    if not (directory / 'completed.json').exists():
        return False
    manifest = json.loads((directory / 'manifest.json').read_text())
    for name, expected in manifest.items():
        path = directory / name
        if not path.resolve().is_relative_to(directory.resolve()):
            raise ValueError('Unsafe run manifest path')
        with path.open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != expected:
            raise ValueError('Run checksum mismatch')
    required = {'settings.json', 'timing.json', 'raw_generations.jsonl',
                directory.name + '.json'}
    if not required <= manifest.keys():
        raise ValueError('Incomplete run manifest')
    return True


def freeze_json(path, value):
    """Keep a plan immutable across sessions."""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError('Frozen plan differs; do not mix experiments')
    else:
        save_json(path, value)
    return value


def finished_attempt(study, phase, mode, item):
    """Find at most one completed attempt for the exact frozen document."""
    root = Path(study) / phase / mode / (
        item['subject'] + '_' + str(item['index'])
    )
    complete = [p for p in sorted(root.glob('attempt_*')) if verify_run(p)]
    if len(complete) > 1:
        raise ValueError('Duplicate completed document')
    if not complete:
        return None
    result = complete[0]
    settings = json.loads((result / 'settings.json').read_text())
    if settings['document_hash'] != item['sha256']:
        raise ValueError('Completed result belongs to another document')
    return result


def run_baseline(lm, manager, directory, selection, session_started):
    """Evaluate fixed questions through harness and retain each completion."""
    from uuid import uuid4

    directory = Path(directory)
    mode = 'chat_5shot_direct'
    scores, invalid, token_caps = [], 0, 0
    for number, item in enumerate(selection['items'], 1):
        completed = finished_attempt(directory, 'baseline', mode, item)
        if completed is None:
            remaining = selection['session_budget_seconds'] - 300
            remaining -= time.monotonic() - session_started
            if remaining <= 0:
                raise TimeoutError('Session budget exhausted; partial ZIP')
            print(f"{number}/{len(selection['items'])}: {item['subject']}",
                  flush=True)
            root = directory / 'baseline' / mode / (
                item['subject'] + '_' + str(item['index'])
            )
            name = 'attempt_' + uuid4().hex[:12]
            lm.expected_input_hash = item.get('input_ids_sha256')
            try:
                run_eval(
                    lm, manager, root, name,
                    'mmlu_prox_ru_chat_' + item['subject'],
                    sample_index=item['index'], document_hash=item['sha256'],
                )
            finally:
                lm.expected_input_hash = None
            completed = root / name
        result = json.loads((completed / (completed.name + '.json'))
                            .read_text())
        records = next(iter(result['samples'].values()))
        if len(records) != 1:
            raise ValueError('Expected one scored question')
        scores.append(records[0]['exact_match'])
        invalid += records[0]['filtered_resps'] == ['[invalid]']
        raw = (completed / 'raw_generations.jsonl').read_text()
        for line in raw.splitlines():
            token_caps += len(json.loads(line)['generated_ids']) >= 2048
    summary = {
        'protocol': selection['protocol'], 'status': 'completed',
        'questions': len(scores), 'correct': sum(scores),
        'accuracy': sum(scores) / len(scores),
        'invalid_answers': invalid, 'outputs_at_token_limit': token_caps,
        'note': 'Mean of saved harness exact_match; not full MMLU-ProX',
    }
    freeze_json(directory / 'summary.json', summary)
    return summary
''',
    'run.py': r'''"""Run the fixed FP8 comparison in an isolated Python process."""

import hashlib
import importlib.metadata
import json
import os
import sys
from pathlib import Path
from uuid import uuid4

os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


def main():
    from baseline import (
        BASE_REVISION,
        HARNESS_REVISION,
        MODEL_ID,
        MODEL_REVISION,
        AuditedVLLM,
        finished_attempt,
        freeze_json,
        run_baseline,
        run_eval,
        save_json,
    )
    from huggingface_hub import hf_hub_download
    from lm_eval.tasks import TaskManager

    root = Path(__file__).parent
    results = Path(sys.argv[1])
    started = float(sys.argv[2])
    budget = float(sys.argv[3])
    reference = json.loads((root / 'nf4_reference.json').read_text())
    selection = dict(reference['selection'])
    selection.update(
        protocol='chat-5shot-fp8-v1', model_revision=MODEL_REVISION,
        items=reference['records'], session_budget_seconds=budget,
        question_budget_seconds=None,
        sources=json.loads((root / 'source_manifest.json').read_text()),
        backend='vllm', precision='FP8 W8A16 Marlin',
    )
    freeze_json(results / 'selection.json', selection)
    provenance = json.loads(importlib.metadata.distribution('lm-eval')
                            .read_text('direct_url.json'))
    if provenance.get('vcs_info', {}).get('commit_id') != HARNESS_REVISION:
        raise RuntimeError('Harness revision mismatch')
    model_sources = {}
    for repo, revision in [('Qwen/Qwen3.8-27B', BASE_REVISION),
                           (MODEL_ID, MODEL_REVISION)]:
        files = {}
        for name in ('config.json', 'tokenizer.json',
                     'tokenizer_config.json', 'chat_template.jinja'):
            path = Path(hf_hub_download(repo, name, revision=revision))
            files[name] = hashlib.sha256(path.read_bytes()).hexdigest()
        model_sources[repo] = files
    expected = json.loads((root / 'model_sources.json').read_text())
    if model_sources != expected:
        raise RuntimeError('Model metadata differs from reviewed sources')
    freeze_json(results / 'model_sources.json', model_sources)
    manager = TaskManager(include_path=str(root / 'lm_eval_tasks'))
    print('Loading FP8 model once; A100 uses W8A16 Marlin.', flush=True)
    lm = AuditedVLLM(
        pretrained=MODEL_ID, revision=MODEL_REVISION,
        tokenizer='Qwen/Qwen3.8-27B', tokenizer_revision=BASE_REVISION,
        dtype='bfloat16', quantization='fp8', batch_size=1,
        max_model_len=32768, max_gen_toks=2048, max_num_seqs=1,
        max_num_batched_tokens=2048, gpu_memory_utilization=0.90,
        enforce_eager=True, enable_prefix_caching=False,
        language_model_only=True, seed=1234, enable_thinking=False,
    )
    smoke = selection['excluded_development'][0]
    smoke_result = run_eval(
        lm, manager, results / 'preflight', 'calibration_' + uuid4().hex[:8],
        'mmlu_prox_ru_chat_' + smoke['subject'], smoke['index'],
        smoke['sha256'],
    )
    smoke_sample = next(iter(smoke_result['samples'].values()))[0]
    if smoke_sample['filtered_resps'] == ['[invalid]']:
        raise RuntimeError('Calibration produced no extractable answer')
    summary = run_baseline(lm, manager, results, selection, started)
    rows = []
    for item in selection['items']:
        completed = finished_attempt(
            results, 'baseline', 'chat_5shot_direct', item,
        )
        path = completed / (completed.name + '.json')
        samples = json.loads(path.read_text())['samples']
        sample = next(iter(samples.values()))[0]
        rows.append({
            'subject': item['subject'], 'question_id': item['question_id'],
            'nf4_correct': item['nf4_correct'],
            'fp8_correct': sample['exact_match'],
            'nf4_answer': item['nf4_answer'],
            'fp8_answer': sample['filtered_resps'][0],
        })
    comparison = {
        'questions': len(rows), 'nf4_correct': 21,
        'fp8_correct': summary['correct'],
        'delta_accuracy_pp': 100 * (summary['accuracy'] - 21 / 28),
        'improved': sum(r['fp8_correct'] > r['nf4_correct'] for r in rows),
        'regressed': sum(r['fp8_correct'] < r['nf4_correct'] for r in rows),
        'nf4_archive_sha256': reference['archive_sha256'],
        'rows': rows,
        'note': 'Historical comparison; backend differs. No speedup claim.',
    }
    save_json(results / 'comparison.json', comparison)
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
    print(json.dumps({k: v for k, v in comparison.items() if k != 'rows'},
                     ensure_ascii=False, indent=2), flush=True)


if __name__ == '__main__':
    main()
''',
    'model_sources.json': r'''{
  "Qwen/Qwen3.8-27B": {
    "config.json": "191e0af232104ed8b65258cf3fb2b842e288008baca7633c11b82a1ac7203aab",
    "tokenizer.json": "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3",
    "tokenizer_config.json": "b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27",
    "chat_template.jinja": "c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041"
  },
  "Qwen/Qwen3.8-27B-FP8": {
    "config.json": "74227dd615bf1ea975aa676bdf355a0379858c12f394b5365cd9dfa5fc2c70bc",
    "tokenizer.json": "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3",
    "tokenizer_config.json": "b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27",
    "chat_template.jinja": "c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041"
  }
}
''',
    'protocol.md': r'''# FP8 baseline

A100 40 GB, Python 3.13. Qwen3.8-27B-FP8, vLLM W8A16 Marlin.
Same 28 NF4 questions, chat 5-shot, thinking off, greedy, max 2048.
One reserved calibration question precedes evaluation; not part of score.
Exact prompt token hashes must match NF4. No truncation or CPU offload.
Historical comparison includes a backend change; no precision-only speedup.
Two-hour operational cap by default, including installation and downloads.
Partial results export on failure. Runtime deletion loses unsaved files.
No claim of GPU verification before the user runs this notebook.

Diagnostic callable RPC removed after the first Colab failure.
Kernel selection and model memory remain in the standard vLLM log.
Worker PyTorch peak memory is not measured by this version.
''',
    'source_manifest.json': r'''{
  "lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml": "852bf00302ac68b796a07121646afeb1c054f165b1b7e1585c89a7a549193669",
  "lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml": "ba71b82a8872d81e841ca566cf1228fbf6bac288963c40669112c4d241156093",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml": "5c5652af779b6f76263c8b2c50d1e556b49ed9ae135a569de7a729f819d2db23",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml": "c3ee2987b8742f81fa4eb59262d3bd247a65efa6a9cb2a75ed7d75427bf08d75",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml": "5be779e17271bb81d0ebe8702a24c3ac399287d0df4836ab81524ea6a098257a",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml": "1816af75a73635c7705c35aef3072f15a22ee54aa305819880927dcf7e3a9ae5",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml": "ef1fcf83eadc67e89bb3b4381bad0eac16585c3bfc266bf5d124f99d4030d487",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml": "fe1bcfa0009c42c7998c37ede092bcf3bb9b896dbbc5885980f92a89ea8f05b5",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml": "80e6d0e0613752a2454df265c58f34f9448c4791e347893abda72305e6fbc64a",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml": "047c600f10c0b38e0b3eff2c428079793c3d89ecd2bcc1b43019df6c83f84956",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml": "7679544fafe2ada9c1e667e4adc13db1f319f420ecb97908ef48198737ba2289",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml": "af96df4b28b32daa1846bc0f29264709ebe2e7a7c4789f137fa900ae61d49694",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml": "87a294c66ccf38044a6016ad0ae0c5cb5a4c3f092ebd2c40dbdc6c4de3ff1580",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml": "fb0599414d22276942170f10fc4238a590e4981d04be08ceae9202af54fe0fe2",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml": "bfb64e6779bef91b5092dea22efee61f8ebf1e7888861558bd308a3a6d4e7196",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml": "cb4802cc2f6f07283d8a433cd55512d436a34d653d6ee5d09f1e0cf561e6a751",
  "lm_eval_tasks/mmlu_prox_ru_chat/utils.py": "99d10756ca88eee8fa3ff9fce3a4d4efb746b17a7a9b9feb7fcfbb76b15e5f3a",
  "nf4_reference.json": "500024adaad1814e3e38ba737ec37ebd19010f73055cb4b4ded25262d76fcfbc",
  "requirements.txt": "0d8268adba83ce6c9062c5085e76c516a7e0b79e33791e33dd7686a452708052",
  "baseline.py": "11a0b7d0d8c61e04c853e899d0db27bce5cbb993270f477215bc94a19362668f",
  "run.py": "ece2f0033cbae9db06a8aa69f4daf893c6d3ccd1829c160bcc285c824f8aa0d9",
  "model_sources.json": "ce5701c04db49306083db628881b4da6fa630c518ddcfec395f43d7505f18f87",
  "protocol.md": "439d1e186de9150c614c7fbce8eb21231574b04da2b3579f155bdaf388f4247e"
}
''',
}
for name, source in SOURCES.items():
    path = RUN_ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text() != source:
        raise RuntimeError("Исходники изменились: " + name)
    if not path.exists():
        path.write_text(source, encoding="utf-8")
for name, digest in json.loads(SOURCES["source_manifest.json"]).items():
    if hashlib.sha256((RUN_ROOT / name).read_bytes()).hexdigest() != digest:
        raise RuntimeError("Ошибка SHA-256: " + name)


In [ ]:
def export_results():
    archive = WORK_ROOT / (RUN_ROOT.name + "_" + uuid4().hex[:8] + ".zip")
    hashes = {}
    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as output:
        paths = [RUN_ROOT / name for name in SOURCES]
        paths += [p for p in RESULTS_DIR.rglob("*") if p.is_file()]
        for path in sorted(paths):
            name = str(path.relative_to(RUN_ROOT))
            data = path.read_bytes()
            hashes[name] = hashlib.sha256(data).hexdigest()
            output.writestr(name, data)
        output.writestr("export_manifest.json", json.dumps({
            "schema_version": 1, "files": hashes,
        }, indent=2))
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    print("ZIP:", archive, "SHA-256:", digest)
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as error:
        print("Скачайте ZIP через Files:", archive, error)
    return archive


def run_command(args, log_name):
    remaining = SESSION_BUDGET_HOURS * 3600
    remaining -= time.monotonic() - SESSION_STARTED + 120
    if remaining <= 0:
        raise TimeoutError("Бюджет сессии исчерпан")
    log_path = RESULTS_DIR / log_name
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            args, cwd=RUN_ROOT, stdout=stream, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        try:
            deadline = time.monotonic() + remaining
            position = 0
            while True:
                left = deadline - time.monotonic()
                if left <= 0:
                    raise TimeoutError("Бюджет сессии исчерпан")
                try:
                    code = process.wait(timeout=min(20, left))
                    break
                except subprocess.TimeoutExpired:
                    with log_path.open() as reader:
                        reader.seek(position)
                        print(reader.read()[-2000:], end="", flush=True)
                        position = reader.tell()
        finally:
            # Terminate the worker group too, including vLLM children.
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        if code:
            print(log_path.read_text()[-5000:])
            raise RuntimeError(f"Ошибка процесса; см. {log_path}")
    print(log_path.read_text()[-5000:])


## Установка и проверка GPU
Зависимости изолированы от пакетов Colab.


In [ ]:
try:
    gpu = subprocess.check_output([
        "nvidia-smi", "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ], text=True)
    print(gpu)
    fields = gpu.strip().split(",")
    if len(fields) != 3 or "A100" not in fields[0]:
        raise RuntimeError("Нужна одна A100 40 GB")
    if not 35000 < int(fields[1]) < 45000:
        raise RuntimeError("Нужна A100 40 GB")
    (RESULTS_DIR / "gpu.txt").write_text(gpu)
    ENV_PYTHON = WORK_ROOT / "venv" / "bin" / "python"
    if not ENV_PYTHON.exists():
        venv.EnvBuilder(with_pip=False).create(ENV_PYTHON.parent.parent)
    print("Установка; лог: results/install.log. Ожидайте несколько минут.")
    run_command([
        sys.executable, "-m", "pip", "--python", str(ENV_PYTHON),
        "install", "-r", str(RUN_ROOT / "requirements.txt"),
    ], "install.log")
    run_command([
        sys.executable, "-m", "pip", "--python", str(ENV_PYTHON), "check",
    ], "pip_check.log")
    run_command([
        sys.executable, "-m", "pip", "--python", str(ENV_PYTHON),
        "freeze", "--all",
    ], "packages.txt")
except BaseException:
    export_results()
    raise


## Проверка и 28 вопросов
Одна загрузка модели, один calibration-вопрос, затем весь baseline.
Результаты сравнения с NF4 — в `comparison.json`. Лог — `results/run.log`.


In [ ]:
try:
    if not (RESULTS_DIR / "comparison.json").exists():
        run_command([
            str(ENV_PYTHON), str(RUN_ROOT / "run.py"), str(RESULTS_DIR),
            str(SESSION_STARTED), str(SESSION_BUDGET_HOURS * 3600),
        ], "run.log")
    comparison = json.loads((RESULTS_DIR / "comparison.json").read_text())
    print(json.dumps({k: v for k, v in comparison.items() if k != "rows"},
                     ensure_ascii=False, indent=2))
finally:
    archive = export_results()


Повторное скачивание при необходимости: `export_results()`.
